## **Trabalho: Estratégias de busca adversarial e Árvores de Decisão**

Trabalho feito por: 

-David Rodrigues 202303949

-Inês Castro 202304060

-Soraia Costa 202305078

**Índice**

**Problema**

O quatro em linha é um jogo bastante simples: basicamente, temos um tabuleiro de 6 linhas por 7 colunas e dois jogadores com peças de cor diferente que colocam uma peça por jogada numa das colunas ainda disponíveis. O objetivo é fazer um quatro em linha, seja, verticalmente, horizontalmente ou diagonalmente. 
No âmbito da UC de IA foi-nos proposto elaborar todo o código que permite efetivamente jogar o quatro em linha e os algoritmos: Monte Carlo Tree Search (MCTS) e Decision Trees utilizando o procedimento ID3 para que deste modo, consigamos apresentar os seguintes cenários: humano vs humano; humano vs computador e computador vs computador, neste caso, Monte Carlo vs ID3.

### **Introdução:**

O jogo Connect Four é um clássico exemplo de jogo adversarial, amplamente utilizado como estudo de caso em inteligência artificial devido à sua simplicidade nas regras e profundidade estratégica. Jogadores colocam peças à vez numa grelha vertical, com o objetivo de alinhar quatro das suas peças em linha reta — na horizontal, vertical ou diagonal. Apesar de fácil de aprender, a complexidade estratégica do jogo oferece um excelente campo de aplicação para técnicas avançadas de tomada de decisão.

Neste trabalho, exploramos duas abordagens distintas para criar agentes capazes de jogar Connect Four de forma competitiva: Monte Carlo Tree Search (MCTS) e árvores de decisão (ID3). 

A MCTS é uma técnica baseada em simulações aleatórias para explorar o espaço de estados do jogo, utilizando uma fórmula de seleção conhecida como Upper Confidence Bound for Trees (UCT). Esta abordagem tem demonstrado grande eficácia em jogos com espaços de decisão extensos e sem heurísticas claras.

Complementarmente, aplicamos o algoritmo ID3 para construir uma árvore de decisão capaz de prever o próximo movimento a partir de um estado de jogo. Para isso, precisamos de gerar um conjunto de dados utilizando o MCTS como base de conhecimento, permitindo assim treinar um modelo supervisionado. Essa combinação entre um método de busca e um método de aprendizagem resulta numa solução híbrida, que une exploração estatística e generalização a partir de exemplos.

Ao longo deste projeto, desenvolvemos um sistema capaz de jogar contra humanos ou contra outros algoritmos, analisando o desempenho de cada abordagem e os desafios envolvidos na sua implementação.

**Solução**

Para solucionar o problema acima descrito optamos por elaborar vários ficheiros com objetivos distintos. Basicamente, temos o ConnectState.py, que define o tabuleiro e as funções básicas que utilizaremos para efetivamente jogar, que serão indespensáveis nos restantes ficheiros; o mcts.py, que define o algoritmo Monte Carlo, recorrendo às funçoes da class ConnectState do ficheiro anterior; o id3_model.py, que define o algoritmo ID3, recorrendo a funçoes da class ConnectState; e o ficheiro Game.py que define a interface e faz a ligação entre a interface e os restantes ficheiros. Seguidamente iremos analisar mais concretamente cada um destes ficheiros.

Bibliotecas utilizadas:

In [1]:
import math, time, sys, random
import pygame
import pandas as pd
import numpy as np
from collections import Counter
from copy import deepcopy

pygame 2.6.1 (SDL 2.28.4, Python 3.11.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


### **Connect State:**

No ConnectState.py definimos duas classes:

A GameMeta, que define os básicos, ou seja, as dimensões do jogo, as constantes que iremos utilizar para representar cada jogador e cada estado do jogo:

In [2]:
class GameMeta:  # Define o jogo, isto é, as constantes usadas pelo jogo: jogadores, resultados, dimensões do tabuleiro...
    PLAYERS = {'none': 0, 'one': 1, 'two': 2} #Quando dada celula='0' quer dizer que está vazia, se celula='1', quer dizer que essa celula tem uma peça do jogador 1 e igualmente para o '2'.
    OUTCOMES = {'none': 0, 'one': 1, 'two': 2, 'draw': 3} #Define o estado do jogo: 'none' se o jogo ainda não tiver acabado, 'one' se o jogador 1 ganhou, 'two' se o jogador 2 ganhou e 'draw' se empataram
    INF = float('inf')
    ROWS = 6   #Vamos definir o tabuleiro como uma matriz com 6 linhas e 7 colunas
    COLS = 7

A ConnectState, que define as mais diversas funçoes do jogo: __init__ (self); get_board(self); move(self,col); get_legal_moves(self); check_win(self); check_win_from(self,row,col); game_over(self); get_outcome(self); clone(self); print(self) que iremos esclarecer mais à frente.

In [3]:
class ConnectState:
    def __init__(self):
        self.board = [[0] * GameMeta.COLS for _ in range(GameMeta.ROWS)] #inicializa o tabuleiro 6*7 com '0' em todas as posiçoes. A board numera as celulas de cima para baixo e da direita para a esquerda.
        self.to_play = GameMeta.PLAYERS['one'] #começa o jogador 1 a jogar
        self.height = [GameMeta.ROWS - 1] * GameMeta.COLS #guarda a linha disponível mais baixa (de baixo para cima) de cada coluna
        self.last_played = [] #guarda o último movimento feito(linha e coluna)

    def get_board(self): #retorna uma copia do tabuleiro
        return deepcopy(self.board)

    def move(self, col):  #Aplica o movimento ao tabuleiro existente 
        self.board[self.height[col]][col] = self.to_play #atualiza o tabuleiro: a linha mais baixa da coluna selecionada passa de '0' para '1' ou '2', dependendo de quem está a jogar
        self.last_played = [self.height[col], col] #guarda o movimento no last_played como último mov
        self.height[col] -= 1 #atualiza a 'altura' da coluna em que jogou
        self.to_play = GameMeta.PLAYERS['two'] if self.to_play == GameMeta.PLAYERS['one'] else GameMeta.PLAYERS['one'] #Passa a vez para o outro jogador

    def get_legal_moves(self):  #Retorna as colunas onde é possível jogar, ou seja, colunas cuja última linha='0'(primeira linha da matriz vazia)
        return [col for col in range(GameMeta.COLS) if self.board[0][col] == 0]

    def check_win(self): #retorna o jogador que ganhou se houver vitória após o último movimento
        if len(self.last_played) > 0 and self.check_win_from(self.last_played[0], self.last_played[1]):
            return self.board[self.last_played[0]][self.last_played[1]]   #retorna o jogador que ganhou, ou seja, o numero presente na celula da ultima jogada.
        return 0

    def check_win_from(self, row, col): #verifica se na celula (row, col) há algum 4 em linha do jogador que está a jogar, 
        player = self.board[row][col]
        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]#verificando a vertical, a horizontal e as diagonais que passam nessa celula. 
        for dr, dc in directions:
            count = 1
            for i in [1, -1]:
                r, c = row, col
                while True:
                    r += i * dr
                    c += i * dc
                    if 0 <= r < GameMeta.ROWS and 0 <= c < GameMeta.COLS and self.board[r][c] == player:
                        count += 1
                    else:
                        break
            if count >= 4: # Se o numero de peças alinhadas (verticalmente, horizontalmente ou diagonalmente) for >=4, o check_win_from retorna true, ou seja, o jogador ganhou
                return True
        return False
    
    def game_over(self):  #Verifica se o jogo acabou, ou seja,  verifica se alguém ganhou ou se já não há mais jogadas possíveis (empate)
        return self.check_win() or len(self.get_legal_moves()) == 0

    def get_outcome(self): #Retorna o resultado do jogo: vitória do jogador 1, vitória do jogador 2 ou empate
        if len(self.get_legal_moves()) == 0 and self.check_win() == 0: #se não há jogadas possíveis e não houve nenhum vencedor até ao momento, retorna empate
            return GameMeta.OUTCOMES['draw']
        return GameMeta.OUTCOMES['one'] if self.check_win() == GameMeta.PLAYERS['one'] else GameMeta.OUTCOMES['two']
    
    def clone(self): #clona o estado atual do tabuleiro
        new_state = ConnectState()
        new_state.board = [row[:] for row in self.board]
        new_state.to_play = self.to_play
        new_state.height = self.height[:]  # acrescentei isto e a linha a seguir
        new_state.last_played = self.last_played[:]
        return new_state


    def print(self): #imprime o tabuleiro no terminal,  usando 'X' para o jogador 1, 'O' para o jogador 2 e espaço em branco para posições vazias
        print('=============================')
        for row in range(GameMeta.ROWS):
            for col in range(GameMeta.COLS):
                print('| {} '.format('X' if self.board[row][col] == 1 else 'O' if self.board[row][col] == 2 else ' '), end='')
            print('|')
        print('=============================')


### **Monte Carlo Tree Search:**

MCTS combines the standards of Monte Carlo strategies, which rely upon random sampling and statistical evaluation, with tree-primarily based search techniques. 
$$
UCT = \frac{Q}{N} + c\sqrt{\frac{In{N_{parent}}}{N}}
$$

A classe Node representa um nó da árvore com estatísticas de visitas e recompensas. A classe MCTS gere a árvore: seleciona nós (select_node), expande (expand), simula jogos aleatórios (roll_out), e atualiza os valores com backpropagation (back_propagate) para escolher a melhor jogada (best_move).

In [4]:
EXPLORATION = math.sqrt(2) #constante utilizada no UCT
PLAYERS = {'none': 0, 'one': 1, 'two': 2} #representa a ausencia de jogador e os jogadores
OUTCOMES = {'none': 0, 'one': 1, 'two': 2, 'draw': 3} #possíveis desfechos do jogo
INF = float('inf') #representa o infinito
ROWS = 6 #número de linhas do tabuleiro
COLS = 7 #número de colunas do tabuleiro

In [5]:
class Node:
    def __init__(self,move,parent):
        self.move = move #movimento que levou a este nó
        self.parent = parent #nó pai
        self.N=0  #número de visitas ao nó
        self.Q=0  #soma dos valores deste nó
        self.children = {} #dicionário dos filhos (movimento -> nó)
        self.outcome = PLAYERS['none'] #resultado do jogo neste nó se conhecido

    #adiciona filhos ao nó atual
    def add_children (self, children : dict)-> None : 
        for child in children:
            self.children[child.move]=child   
    
    def value (self, explore : float = EXPLORATION): #IMPORTANTE ELE DIZ Q ISTO NAO ESTA DEFINIDO MAS CORRE
        #calcula o valor do nó usando a fórmula do UCT
        if self.N == 0:
            return 0 if explore==0 else INF  #se ainda não foi visitado, retorna infinito se explorar, senão 0
        else:
            return self.Q / self.N + explore* math.sqrt(math.log(self.parent.N)/ self.N)  #UCT fórmula

class MCTS:
    def __init__(self, state = ConnectState()):
        self.root_state = deepcopy(state) #estado do jogo na raiz da árvore 
        self.root = Node (None,None) #nó raiz 
        self.run_time = 0 #tempo de execucao da pesquisa
        self.node_count = 0 #total de nós visitados
        self.num_rollouts = 0 #número de simulações realizadas
    
    def select_node(self) -> tuple:
        node = self.root #seleciona o nó com maior UCT, expandindo se necessário
        state = deepcopy(self.root_state)

        while len(node.children) !=0:
            children = node.children.values()
            max_value = max(children, key = lambda n : n.value()).value() #maior valor entre os filhos
            max_nodes = [n for n in children if n.value() == max_value]   #filhos com valor máximo
            node = random.choice(max_nodes) #escolher aleatoriamente entre os melhores
            state.move(node.move)  
            if node.N == 0:    
                return node,state   
        if self.expand (node , state): #expande o nó só se possível
            node = random.choice(list(node.children.values()))     
            state.move(node.move) # aplica a jogado ao estado
        return node, state
    
    def expand (self, parent : Node, state : ConnectState) -> bool:
        if state.game_over(): # se o jogo terminou, não há expansão possível
            return False  
        # Cria novos nós para cada jogada legal a partir do estado atual
        children = [Node(move,parent)for move in state.get_legal_moves()]
        parent.add_children(children)
        return True
    
    def roll_out (self , state : ConnectState)-> int: 
        while not state.game_over():
            state.move(random.choice(state.get_legal_moves())) #simula um jogo aleatório até ao fim a partir do estado atual
        return state.get_outcome #retornas o resultado da simulacao
    
    def back_propagate (self,node : Node, turn: int, outcome:int)-> None : 
        reward = 0 if outcome == turn else 1 #cacula a recompensa inicial (0 se vitória, 1 se derrota, depois alterna)
        while node is not None:
            node.N +=1 
            node.Q +=reward
            node = node.parent
            if outcome==OUTCOMES['draw']:
                reward = 0
            else:
                reward = 1-reward

    def search(self, time_limit : int):
        start_time = time.process_time()  #marca o início da pesquisa
        num_rollouts =0
        while time.process_time() - start_time < time_limit:
            node, state = self.select_node()       #seleciona um nó a explorar
            outcome = self.roll_out(state)          #executa uma simulacão até ao fim
            self.back_propagate(node,state.to_play, outcome) #atualiza os valores da árvore
            num_rollouts+=1
        run_time = time.process_time()-start_time   #algumas estatisticas
        self.run_time = run_time
        self.num_rollouts = num_rollouts

    def best_move(self):
        if self.root_state.game_over():
            return -1 #  Se o jogo acabou, não há movimento possível
        max_value = max(self.root.children.values(), key = lambda n:n.N).N  # Procura o número máximo de visitas entre os filhos da raiz
        max_nodes = [n for n in self.root.children.values() if n.N==max_value]  # Lista de filhos com o mesmo número máximo de visitas
        best_child = random.choice(max_nodes) # Escolhe aleatoriamente entre os melhores filhos
        return best_child.move # Retorna a jogada correspondente ao melhor filho
    
    def move(self, move):
        if move in self.root.children:  # Se o movimento já está na árvore
            self.root_state.move(move)  # Atualiza o estado do jogo
            self.root = self.root.children[move]  # Avança para o nó correspondente
            return
        self.root_state.move(move)   # Caso o movimento não esteja na árvore, apenas atualiza o estado
        self.root = Node(None,None)  # E cria uma nova raiz sem filhos (nova árvore MCTS a partir deste estado)

    # Retorna o número de simulações realizadas e o tempo total da pesquisa
    def statistics(self)-> tuple:
        return self.num_rollouts, self.run_time   
    
    def move_statistics(self):
        return {move: (child.N, child.Q) for move, child in self.root.children.items()}# Retorna um dicionário com estatísticas de cada movimento possível a partir do estado atual
    # Formato: {movimento: (número de visitas, soma das recompensas)}



### **ID3 procedure:**

Gerar o dataset:

Usa um dataset CSV gerado por MCTS para treinar a árvore com base na informação dos 7 estados de coluna. A função predict_id3_move converte o tabuleiro para o formato do dataset, classifica com a árvore, e retorna uma jogada válida.

In [6]:
import pandas as pd
import math
import random
from collections import Counter


# === Carregar dataset e construir árvore ===
data = pd.read_csv("connect4_mcts_dataset_clean.csv")
features = [f'col_{i}' for i in range(7)]
target_attribute = 'suggested_move'

# === Entropia ===
def entropy(labels):
    total = len(labels)
    counts = Counter(labels)
    return -sum((count / total) * math.log2(count / total) for count in counts.values() if count > 0)

# === Ganho de informação ===
def information_gain(data, split_attribute, target_attribute='suggested_move'):
    total_entropy = entropy(data[target_attribute])
    values = data[split_attribute].unique()
    subset_entropy = 0.0
    for value in values:
        subset = data[data[split_attribute] == value]
        weight = len(subset) / len(data)
        subset_entropy += weight * entropy(subset[target_attribute])
    return total_entropy - subset_entropy

# === Algoritmo ID3 ===
def id3(data, features, target_attribute='suggested_move'):
    labels = data[target_attribute]
    if len(set(labels)) == 1:
        return labels.iloc[0]
    if len(features) == 0:
        return labels.mode()[0]
    gains = [(attr, information_gain(data, attr, target_attribute)) for attr in features]
    best_attr = max(gains, key=lambda x: x[1])[0]
    tree = {best_attr: {}}
    for value in data[best_attr].unique():
        subset = data[data[best_attr] == value]
        if subset.empty:
            tree[best_attr][value] = labels.mode()[0]
        else:
            remaining_features = [f for f in features if f != best_attr]
            subtree = id3(subset, remaining_features, target_attribute)
            tree[best_attr][value] = subtree
    return tree

# === Classificação ===
def classify(tree, example):
    if not isinstance(tree, dict):
        return tree
    attribute = next(iter(tree))
    value = example.get(attribute)
    if value in tree[attribute]:
        return classify(tree[attribute][value], example)
    else:
        return None

# === Conversão do tabuleiro para formato do dataset ===
def board_to_columns(board):
    return {
        f'col_{c}': "".join(str(board[r][c]) for r in range(5, -1, -1)) for c in range(7)
    }

# === Treinar a árvore (feito uma vez ao carregar o módulo) ===
decision_tree = id3(data, features, target_attribute)

# === Função a ser usada no Game.run() ===
def predict_id3_move(state: ConnectState):
    board = state.get_board()
    example = board_to_columns(board)
    move = classify(decision_tree, example)

    # Fallback caso jogada não seja válida
    legal = state.get_legal_moves()
    if move in legal:
        return move
    elif legal:
        return random.choice(legal)
    else:
        return None


### Minimax

In [7]:
from ConnectState import ConnectState, GameMeta
import math
from copy import deepcopy


class minimax:
    def __init__(self, state, max_depth=4):
        self.root_state = deepcopy(state)
        self.max_depth = max_depth

    def evaluate (self, state):
        #estados finais
        if state.game_over():
            if state.check_win() == GameMeta.PLAYERS['one']:
                return math.inf
            elif state.check_win() == GameMeta.PLAYERS['two']:
                return -math.inf
            return 0
        
        #heuristica simples com preferencia central para quando nao é um estado final 

        score = 0
        center_col = GameMeta.COLS // 2

        for col in range(GameMeta.COLS):
            for row in range(GameMeta.ROWS):
                if state.board[row][col] == GameMeta.PLAYERS['one']:
                    score -= 0.5*(center_col - abs(col - center_col))
                elif state.board[row][col] == GameMeta.PLAYERS['two']:
                    score += 0.5*(center_col - abs(col - center_col))
        
        #avaliacao potencial de linhas

        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]  #horizontal, vertical e diagonal

        for row in range(GameMeta.ROWS):
            for col in range(GameMeta.COLS):
                if state.board[row][col] != 0:  #so avaliar posicoes ocupadas
                    player = state.board[row][col]
                    for dr, dc in directions:
                        length = 1
                        empty_spaces = 0
                        for i in range(1, 4):
                            r = row + i*dr
                            c = col + i*dc
                            if 0 <= r < GameMeta.ROWS and 0 <= c < GameMeta.COLS:
                                if state.board[r][c] == player:
                                    length += 1
                                elif state.board[r][c] == 0: # se for jogavel
                                    if r == GameMeta.ROWS-1 or state.board[r+1][c] != 0:
                                        empty_spaces += 1
                                else: #esta bloquado
                                    break  
                        
                        #score baseado no valor potencial das linhas
                        if length >= 2:
                            value = 10 ** (length - 1)  # 2 -> 10 , 3->100
                            if empty_spaces >= (4 - length):
                                if player == GameMeta.PLAYERS['one']:
                                    score += value
                                else:
                                    score -= value

        return float(score)
    
    def minimax(self, state, depth, is_maximizing): #minimax puro com recursao
        if depth==0 or state.game_over():
            return self.evaluate(state)
        legal_moves = state.get_legal_moves()
        best_value = -math.inf if is_maximizing else math.inf

        for move in legal_moves:
            child = deepcopy(state)
            child.move(move)
            val = self.minimax(child, depth-1, not is_maximizing)

            if is_maximizing:
                best_value = max(best_value, val)
            else:
                best_value = min(best_value, val)
        return best_value

    def best_move(self): ##prioriza moves vencedores, bloquear oponente e centro/valor potencial de linhas
        legal_moves = self.root_state.get_legal_moves()
        if not legal_moves:
            return None
        
        #winning move
        
        for move in legal_moves:
            child = deepcopy(self.root_state)
            child.move(move)
            if child.check_win() == self.root_state.to_play:
                return move

        #bloquear vitoria de oponente

        opponent = 3 - self.root_state.to_play
        for move in self.root_state.get_legal_moves():
            opponent_state = deepcopy(self.root_state)
            opponent_state.to_play = opponent  # Switch to opponent's turn
            opponent_state.move(move)
            if opponent_state.check_win() == opponent:
                return move

        #procura normal de minimax

        best_move = None
        best_value = -math.inf

        for move in legal_moves:
            child = deepcopy(self.root_state)
            child.move(move)
            val = self.minimax(child,self.max_depth-1, False)
            if val>best_value:
                best_value = val
                best_move=move
        return best_move

### **Game:**

O código cria uma interface gráfica com pygame para jogar Connect Four, com menu inicial e tabuleiro interativo. O método Game.run() processa cliques do rato, atualiza o estado do jogo (ConnectState) e chama MCTS ou ID3 consoante o modo. A função desenhar_estatisticas_mcts() exibe um gráfico com número de simulações e vitórias para cada coluna sugerida pelo MCTS.

In [ ]:
class GameMeta:
    PLAYERS = {'none': 0, 'one': 1, 'two': 2}
    OUTCOMES = {'none': 0, 'one': 1, 'two': 2, 'draw': 3}
    INF = float('inf')
    ROWS = 6
    COLS = 7

class Game:
    AZUL = (100, 149, 237)
    ROSA = (255, 182, 193)
    AMARELO = (255, 255, 153)
    BRANCO = (255, 255, 255)
    PRETO = (0,0,0)

    TAMANHO_QUADRADO = 100
    RAIO = int(TAMANHO_QUADRADO / 2 - 5)

    def __init__(self):
        self.estado = ConnectState()
        self.jogo_acabou = False
        self.modo = None
        self.mcts = MCTS(self.estado)
        pygame.init()
        self.grafico_largura = 600
        self.largura = GameMeta.COLS * self.TAMANHO_QUADRADO
        self.altura = (GameMeta.ROWS + 1) * self.TAMANHO_QUADRADO
        self.tela = pygame.display.set_mode((self.largura + self.grafico_largura, self.altura))
        self.fonte = pygame.font.SysFont("monospace", 65)
        self.fonte_pequena = pygame.font.SysFont("monospace", 20)
        self.fonte_menu = pygame.font.SysFont("monospace", 30)
        self.menu()
        self.estatisticas_mcts = {}


    def menu(self):
        self.tela.fill(self.BRANCO)
        pvp_text = self.fonte_menu.render("Player vs Player", True, self.PRETO)
        ai_text = self.fonte_menu.render("Player vs MCTS", True, self.PRETO)
        id3_text = self.fonte_menu.render("Player vs ID3", True, self.PRETO)

        self.tela.blit(pvp_text, (self.largura / 2 - pvp_text.get_width() / 2, 150))
        self.tela.blit(ai_text, (self.largura / 2 - ai_text.get_width() / 2, 250))
        self.tela.blit(id3_text, (self.largura / 2 - id3_text.get_width() / 2, 350))

        pygame.display.update()

        while self.modo is None:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    sys.exit()
                if event.type == pygame.MOUSEBUTTONDOWN:
                    x, y = event.pos
                    if 150 <= y <= 200:
                        self.modo = 'pvp'
                    elif 250 <= y <= 300:
                        self.modo = 'ai'
                    elif 350 <= y <= 400:
                        self.modo = 'id3'

        
        self.tela.fill(self.BRANCO)               
        self.desenhar_tabuleiro()

    def desenhar_tabuleiro(self):
        # Draw game board
        for c in range(GameMeta.COLS):
            for l in range(GameMeta.ROWS):
                pygame.draw.rect(self.tela, self.AZUL, (c * self.TAMANHO_QUADRADO, (l + 1) * self.TAMANHO_QUADRADO, self.TAMANHO_QUADRADO, self.TAMANHO_QUADRADO))
                pygame.draw.circle(self.tela, self.BRANCO, (int(c * self.TAMANHO_QUADRADO + self.TAMANHO_QUADRADO / 2), int((l + 1) * self.TAMANHO_QUADRADO + self.TAMANHO_QUADRADO / 2)), self.RAIO)

        # Draw pieces
        board = self.estado.get_board()
        for c in range(GameMeta.COLS):
            for l in range(GameMeta.ROWS):
                if board[l][c] == 1:
                    pygame.draw.circle(self.tela, self.ROSA, (int(c * self.TAMANHO_QUADRADO + self.TAMANHO_QUADRADO / 2), int((l + 1) * self.TAMANHO_QUADRADO + self.TAMANHO_QUADRADO / 2)), self.RAIO)
                elif board[l][c] == 2:
                    pygame.draw.circle(self.tela, self.AMARELO, (int(c * self.TAMANHO_QUADRADO + self.TAMANHO_QUADRADO / 2), int((l + 1) * self.TAMANHO_QUADRADO + self.TAMANHO_QUADRADO / 2)), self.RAIO)
        
        # Draw graph
        self.desenhar_estatisticas_mcts()
        pygame.display.update()
    
    def desenhar_estatisticas_mcts(self):
        if not hasattr(self, 'estatisticas_mcts') or not self.estatisticas_mcts:
            # Clear graph area if no statistics
            pygame.draw.rect(self.tela, self.BRANCO, (self.largura, 0, self.grafico_largura, self.altura))
            return

        graph_x_offset = self.largura + 50
        graph_top = 300
        bar_width = 35
        bar_spacing = 35
        max_bar_height = 200
        label_gap = 5
        text_height = 15

        # Clear graph area
        pygame.draw.rect(self.tela, self.BRANCO, (self.largura, 0, self.grafico_largura, self.altura))

        # Calculate max visits for scaling
        visits = [n for (n, q) in self.estatisticas_mcts.values()]
        max_visits = max(visits) if visits else 1

        # We always want to show all 7 possible columns
        moves_to_display = sorted(self.estatisticas_mcts.items(), key=lambda x: x[0])
        
        for i, (move, (n, q)) in enumerate(moves_to_display):
            # Calculate bar dimensions
            bar_height = int((n / max_visits) * max_bar_height) if max_visits > 0 else 0
            win_ratio = q / n if n > 0 else 0
            win_height = int(win_ratio * bar_height) if bar_height > 0 else 0

            # Calculate positions
            bar_x = graph_x_offset + i * (bar_width + bar_spacing)
            bar_y = graph_top + (max_bar_height - bar_height)

            # Draw total visit bar (blue)
            pygame.draw.rect(self.tela, self.AZUL, (bar_x, bar_y, bar_width, bar_height))

            # Draw win portion (red)
            pygame.draw.rect(self.tela, (255, 0, 0), (bar_x, bar_y + bar_height - win_height, bar_width, win_height))

            # Draw move number below bar (always visible)
            move_text = self.fonte_pequena.render(f"Col {move+1}", True, self.PRETO)
            text_x = bar_x + (bar_width - move_text.get_width()) // 2
            text_y = graph_top + max_bar_height + label_gap
            self.tela.blit(move_text, (text_x, text_y))

            # Draw statistics above the bar
            stats_y = bar_y - 40  # Position above the bar
            
            # Draw wins (W) above bar
            wins_text = self.fonte_pequena.render(f"W:{int(q)}", True, self.PRETO)
            wins_x = bar_x + (bar_width - wins_text.get_width()) // 2
            self.tela.blit(wins_text, (wins_x, stats_y))

            # Draw visits (V) above wins
            visits_text = self.fonte_pequena.render(f"T:{n}", True, self.PRETO)
            visits_x = bar_x + (bar_width - visits_text.get_width()) // 2
            self.tela.blit(visits_text, (visits_x, stats_y - text_height - 2))

            # Draw win percentage below the move label
            win_pct = f"{win_ratio*100:.1f}%" if n > 0 else "0%"
            pct_text = self.fonte_pequena.render(win_pct, True, self.PRETO)
            pct_x = bar_x + (bar_width - pct_text.get_width()) // 2
            pct_y = text_y + move_text.get_height() + 2
            self.tela.blit(pct_text, (pct_x, pct_y))

        # Add title and legend
        title_text = self.fonte_pequena.render("MCTS Move Statistics", True, self.PRETO)
        self.tela.blit(title_text, (graph_x_offset + (self.grafico_largura - 40 - title_text.get_width()) // 2, 20))

        # Add legend
        legend_y = graph_top + max_bar_height + 80
        pygame.draw.rect(self.tela, self.AZUL, (graph_x_offset, legend_y, 15, 15))
        legend_text = self.fonte_pequena.render("Total visits (T)", True, self.PRETO)
        self.tela.blit(legend_text, (graph_x_offset + 20, legend_y))

        pygame.draw.rect(self.tela, (255, 0, 0), (graph_x_offset, legend_y + 20, 15, 15))
        legend_text2 = self.fonte_pequena.render("Winning visits (W)", True, self.PRETO)
        self.tela.blit(legend_text2, (graph_x_offset + 20, legend_y + 20))
    def run(self):
        while not self.jogo_acabou:
            for evento in pygame.event.get():

                if evento.type == pygame.MOUSEMOTION:
                    # Only draw the preview in the game area (not the graph area)
                    if evento.pos[0] < self.largura-50:
                        pygame.draw.rect(self.tela, self.BRANCO, (0, 0, self.largura, self.TAMANHO_QUADRADO))
                        pos_x = evento.pos[0]
                        cor = self.ROSA if self.estado.to_play == 1 else self.AMARELO
                        pygame.draw.circle(self.tela, cor, (pos_x, int(self.TAMANHO_QUADRADO / 2)), self.RAIO)
                        pygame.display.update()

                if evento.type == pygame.MOUSEBUTTONDOWN:
                    # Only process clicks in the game area
                    if evento.pos[0] < self.largura:
                        pygame.draw.rect(self.tela, self.BRANCO, (0, 0, self.largura, self.TAMANHO_QUADRADO))
                        pos_x = evento.pos[0]
                        coluna = int(math.floor(pos_x / self.TAMANHO_QUADRADO))

                        if coluna in self.estado.get_legal_moves():
                            self.estado.move(coluna)
                            self.mcts.move(coluna)
                            self.desenhar_tabuleiro()

                            winner = self.estado.check_win()
                            if winner:
                                texto = self.fonte.render(f"Jogador {winner} venceu!", 1, self.ROSA if winner == 1 else self.AMARELO)
                                self.tela.blit(texto, (40, 10))
                                self.jogo_acabou = True
                            elif self.modo == 'ai' and not self.estado.game_over():
                                self.mcts.search(5)
                                self.estatisticas_mcts = self.mcts.move_statistics()
                                movimento_mcts = self.mcts.best_move()
                                self.estado.move(movimento_mcts)
                                self.mcts.move(movimento_mcts)
                                self.desenhar_tabuleiro()
                                winner = self.estado.check_win()
                                if winner:
                                    texto = self.fonte.render(f"Jogador {winner} venceu!", 1, self.ROSA if winner == 1 else self.AMARELO)
                                    self.tela.blit(texto, (40, 10))
                                    self.jogo_acabou = True
                            
                            elif self.modo == 'id3' and not self.estado.game_over():
                                from id3_model import predict_id3_move  # <-- função que você implementou
                                movimento_id3 = predict_id3_move(self.estado)
                                if movimento_id3 is not None:
                                    self.estado.move(movimento_id3)
                                    self.desenhar_tabuleiro()
                                    winner = self.estado.check_win()
                                    if winner:
                                        texto = self.fonte.render(f"Jogador {winner} venceu!", 1, self.ROSA if winner == 1 else self.AMARELO)
                                        self.tela.blit(texto, (40, 10))
                                        self.jogo_acabou = True


                            if self.jogo_acabou:
                                pygame.display.update()
                                pygame.time.wait(3000)

if __name__ == "__main__":
    game = Game()
    game.run()

KeyboardInterrupt: 

: 

**Fontes**

https://www.geeksforgeeks.org/ml-monte-carlo-tree-search-mcts/
